# Lesson 5.2: The t-Distribution and Intervals for a Mean

Build a t-interval for a mean from nine nitrate readings, and see how the t critical value compares with z for small and larger samples.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. The sample

The lesson's nine nitrate readings (mg/L). Because sigma is unknown we use the sample SD s, and the standard error is s / sqrt(n).

In [1]:
import numpy as np
from scipy import stats

nitrate = np.array([3.1, 4.2, 2.8, 3.9, 4.5, 3.3, 4.1, 3.7, 3.5])
n = len(nitrate)
mean = nitrate.mean()
se = nitrate.std(ddof=1) / np.sqrt(n)

print(f'mean = {mean:.3f}, s = {nitrate.std(ddof=1):.3f}, SE = {se:.3f}')

mean = 3.678, s = 0.554, SE = 0.185


## 2. The t-interval in one line

stats.t.interval takes the confidence level, the degrees of freedom (n minus 1), the centre and the standard error, and returns the lower and upper limits.

In [2]:
low, high = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)

print(f'df = {n - 1}')
print(f'95% CI (t) = ({low:.2f}, {high:.2f}) mg/L')

df = 8
95% CI (t) = (3.25, 4.10) mg/L


## 3. Check it by hand, and compare with z

The interval is mean plus or minus t* times SE. Using z* = 1.96 by mistake gives a narrower interval that claims too much certainty.

In [3]:
t_star = stats.t.ppf(0.975, df=n - 1)
z_star = stats.norm.ppf(0.975)

print(f't* = {t_star:.3f}, margin = {t_star * se:.3f}')
print(f'by hand: ({mean - t_star * se:.2f}, {mean + t_star * se:.2f})')
print(f'with z*: ({mean - z_star * se:.2f}, {mean + z_star * se:.2f})  too narrow')

t* = 2.306, margin = 0.426
by hand: (3.25, 4.10)
with z*: (3.32, 4.04)  too narrow


## 4. t versus z for different sample sizes

The t critical value is larger for small samples, because s is a shaky estimate of sigma. As n grows, t* shrinks towards z* = 1.960.

In [4]:
print('  n   df   t*      z*')
for k in (5, 10, 30):
    print(f'{k:3d}  {k - 1:3d}  {stats.t.ppf(0.975, k - 1):.3f}  {stats.norm.ppf(0.975):.3f}')

  n   df   t*      z*
  5    4  2.776  1.960
 10    9  2.262  1.960
 30   29  2.045  1.960


## Try this

- Change the level to 0.99 in stats.t.interval and predict whether the interval gets wider or narrower.
- Add k = 100 and k = 1000 to the loop and watch t* approach 1.960.
- Add one extreme reading of 9.0 to the data. What happens to the width of the interval?

## The same idea in R, SPSS and Stata

**R**

```r
nitrate <- c(3.1, 4.2, 2.8, 3.9, 4.5, 3.3, 4.1, 3.7, 3.5)
t.test(nitrate)$conf.int
qt(0.975, df = 8)
qt(0.975, df = c(4, 9, 29))
qnorm(0.975)
```

**SPSS** (Analyze > Descriptive Statistics > Explore (Statistics button: Descriptives, 95% confidence interval for the mean); critical values from Transform > Compute Variable with IDF.T)

```
DATA LIST FREE / nitrate.
BEGIN DATA
3.1 4.2 2.8 3.9 4.5 3.3 4.1 3.7 3.5
END DATA.
EXAMINE VARIABLES=nitrate
  /PLOT=NONE
  /STATISTICS=DESCRIPTIVES
  /CINTERVAL 95.
COMPUTE t5 = IDF.T(0.975, 4).
COMPUTE t10 = IDF.T(0.975, 9).
COMPUTE t30 = IDF.T(0.975, 29).
COMPUTE z = IDF.NORMAL(0.975, 0, 1).
LIST VARIABLES=t5 t10 t30 z /CASES=FROM 1 TO 1.
```

**Stata**

```
clear
input nitrate
3.1
4.2
2.8
3.9
4.5
3.3
4.1
3.7
3.5
end
ci means nitrate
display invttail(4, 0.025) " " invttail(9, 0.025) " " invttail(29, 0.025) " " invnormal(0.975)
```